> **Note:** This notebook requires the following dependencies: `shapely`, `httpx`, `matplotlib` (optional, for plots). Run `pip install shapely httpx matplotlib numpy` if missing.

# 4. Polygon → Clipped Satellite Imagery Pipeline

This notebook traces the **full implemented pipeline** from selecting a polygon to receiving clipped satellite imagery — exactly as the `ImageryModal` and `POST /api/imagery/fetch` work in production.

**Pipeline stages:**
1. Load a polygon boundary (e.g., Wajir County, Kenya)
2. Select a satellite source (e.g., Sentinel-2 L2A)
3. STAC query — find scenes matching polygon, date range, cloud cover
4. Coverage validation — verify scene footprints contain the polygon
5. Phase 1 (Preview) — return the raw STAC COG URL for TiTiler rendering
6. Phase 2 (Clipped COG) — build VRT mosaic, gdalwarp clip, upload to GCS
7. Preprocessing audit — what's implemented vs. what's missing

## Setup

In [ ]:
import json
import math
import os
import sys
from typing import Any

# Add project root to path so we can import backend modules
PROJECT_ROOT = os.path.join(os.getcwd(), "..")
sys.path.insert(0, PROJECT_ROOT)

# Shapely for geometry operations
from shapely.geometry import Polygon, shape, mapping
from shapely.ops import unary_union

# Try importing the actual backend COG service
try:
    from backend.engine.service_layer.cog_service import (
        SOURCE_CONFIG,
        _query_stac_scenes,
        _check_scene_coverage,
        get_preview_scene,
        _generate_clipped_cog_sync,
        compute_geojson_area,
        _mosaic_hash,
        Scene,
    )
    COG_SERVICE_AVAILABLE = True
    print("✓ cog_service.py imported successfully")
except ImportError as e:
    COG_SERVICE_AVAILABLE = False
    print(f"✗ cog_service.py import failed: {e}")
    print("  The notebook will use direct STAC API calls instead.")

# HTTP client for direct STAC queries
import httpx

# Try visualization libraries
try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    from matplotlib.patches import Polygon as MplPolygon
    HAS_MPL = True
    print("✓ matplotlib available")
except ImportError:
    HAS_MPL = False
    print("✗ matplotlib not available — text-only output")

try:
    import numpy as np
    HAS_NP = True
    print("✓ numpy available")
except ImportError:
    HAS_NP = False
    print("✗ numpy not available")

print("\nAll imports complete.")

---
## 1. Load Polygon — Wajir County, Kenya

We load Wajir County (admin1, PCode KE008) from the Kenya admin boundary GeoJSON. This is the same data served by the `/admin-data/kenya/admin1` API route.

In [ ]:
ADMIN1_PATH = os.path.join(PROJECT_ROOT, "frontend", "public", "data", "kenya", "admin1.geojson")

with open(ADMIN1_PATH) as f:
    admin1_data = json.load(f)

# Find Wajir (adm1_name == "Wajir")
wajir_feature = None
for feat in admin1_data["features"]:
    props = feat["properties"]
    if props.get("adm1_name") == "Wajir":
        wajir_feature = feat
        break

if not wajir_feature:
    raise ValueError("Wajir not found in admin1.geojson")

wajir_geometry = wajir_feature["geometry"]
wajir_props = wajir_feature["properties"]
wajir_poly = shape(wajir_geometry)

print(f"Name:       {wajir_props.get('adm1_name', 'Wajir')}")
print(f"PCode:      {wajir_props.get('adm1_pcode', 'KE008')}")
print(f"Area:       {wajir_props.get('area_sqkm', '?'):,} km²")
print(f"Geometry:   {wajir_poly.geom_type}")
print(f"Vertices:   {len(wajir_poly.exterior.coords)}")
print(f"Bounds:     {[round(b, 4) for b in wajir_poly.bounds]}")
print(f"Centroid:   ({round(wajir_poly.centroid.x, 4)}, {round(wajir_poly.centroid.y, 4)})")

### Visualize Wajir boundary

In [ ]:
if HAS_MPL:
    fig, ax = plt.subplots(1, 1, figsize=(8, 8))

    # Plot Wajir polygon
    if wajir_poly.geom_type == "MultiPolygon":
        for poly in wajir_poly.geoms:
            xs, ys = poly.exterior.xy
            ax.fill(xs, ys, alpha=0.3, fc="blue", ec="darkblue", lw=2)
    else:
        xs, ys = wajir_poly.exterior.xy
        ax.fill(xs, ys, alpha=0.3, fc="blue", ec="darkblue", lw=2)

    ax.set_title(f"Wajir County — {wajir_props.get('area_sqkm', 0):,.0f} km²")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)
    plt.show()
else:
    print("Text representation of Wajir polygon (first 5 vertices):")
    if wajir_poly.geom_type == "MultiPolygon":
        coords = list(wajir_poly.geoms[0].exterior.coords[:5])
    else:
        coords = list(wajir_poly.exterior.coords[:5])
    for i, (x, y) in enumerate(coords):
        print(f"  ({x:.4f}, {y:.4f})")
    print(f"  ... ({len(list(wajir_poly.exterior.coords))} vertices total)")

---
## 2. Select Satellite Source — Sentinel-2 L2A

This is configured in `SOURCE_CONFIG` in `cog_service.py`. We pick Sentinel-2 L2A (10m, global) which queries `sentinel-2-l2a` on Microsoft Planetary Computer.

In [ ]:
SOURCE_ID = "sentinel-2-l2a"

if SOURCE_ID in SOURCE_CONFIG:
    cfg = SOURCE_CONFIG[SOURCE_ID]
    print(f"Source:       {SOURCE_ID}")
    print(f"Type:         {cfg['type']}")
    print(f"STAC URL:     {cfg['stac_url']}")
    print(f"Collection:   {cfg['collection']}")
    print(f"Asset key:    {cfg['asset_key']}")
    print(f"Cloud field:  {cfg['cloud_cover_field']}")
else:
    print(f"{SOURCE_ID} not found in SOURCE_CONFIG")
    print("Available sources:", list(SOURCE_CONFIG.keys()))

# Filter parameters (same as what ImageryModal sends)
DATE_FROM = "2024-01-01"
DATE_TO = "2024-12-31"
MAX_CLOUD_COVER = 30.0  # percent
BUFFER_METERS = 0  # no buffer for now

print(f"\nFilter params:")
print(f"  Date range:      {DATE_FROM} → {DATE_TO}")
print(f"  Max cloud cover: {MAX_CLOUD_COVER}%")
print(f"  Buffer:          {BUFFER_METERS}m")

---
## 3. Query STAC for Matching Scenes

This is the core of Phase 1. We call `_query_stac_scenes()` — the same function `get_preview_scene()` and `_generate_clipped_cog_sync()` use internally.

It queries Microsoft Planetary Computer's STAC API with:
- `collections=["sentinel-2-l2a"]`
- `bbox` derived from the polygon (with buffer applied)
- `datetime` date range filter
- `query` with `eo:cloud_cover ≤ max_cloud_cover`
- Results sorted by `datetime desc`
- Asset URLs signed via `planetary_computer.sign()`

In [ ]:
def query_stac_scenes_direct(
    source_id: str,
    geometry: dict,
    buffer_meters: int = 0,
    date_from: str | None = None,
    date_to: str | None = None,
    max_cloud_cover: float = 100,
) -> list[dict[str, Any]]:
    """
    Query STAC API directly via HTTP.
    This mirrors what _query_stac_scenes() does in cog_service.py
    but uses raw httpx so we can inspect the actual request/response.
    """
    if source_id not in SOURCE_CONFIG:
        raise ValueError(f"Unknown source '{source_id}'")
    cfg = SOURCE_CONFIG[source_id]

    poly = shape(geometry)
    if buffer_meters > 0:
        poly = poly.buffer(buffer_meters / 111_320.0)
    bbox = list(poly.bounds)

    # Build STAC search parameters
    params: dict[str, Any] = {
        "collections": [cfg["collection"]],
        "bbox": bbox,
        "max_items": 20,
        "sortby": [{"field": "properties.datetime", "direction": "desc"}],
    }

    # Date range
    if date_from and date_to:
        params["datetime"] = f"{date_from}/{date_to}"
    elif date_from:
        params["datetime"] = f"{date_from}/.."
    elif date_to:
        params["datetime"] = f"../{date_to}"

    # Cloud cover filter
    cc_field = cfg.get("cloud_cover_field")
    if cc_field and max_cloud_cover < 100:
        params["query"] = {cc_field: {"lte": max_cloud_cover}}

    print(f"STAC search params:")
    print(json.dumps(params, indent=2))

    # Make the request
    stac_url = cfg["stac_url"]
    search_url = f"{stac_url}/search"
    print(f"\nPOST {search_url}")

    with httpx.Client(timeout=60.0) as client:
        resp = client.post(search_url, json=params)
        resp.raise_for_status()
        data = resp.json()

    features = data.get("features", [])
    print(f"Found {len(features)} scenes")

    scenes = []
    for item in features:
        props = item.get("properties", {})
        assets = item.get("assets", {})

        # Find the right asset (same fallback chain as cog_service)
        asset_key = cfg["asset_key"]
        asset = assets.get(asset_key)
        if not asset:
            for key in ("image", "cog", "visual", "rendered_preview", "B04", "B05", "data"):
                if key in assets:
                    asset = assets[key]
                    break
        if not asset:
            continue

        cc = props.get("eo:cloud_cover")
        gsd = props.get("gsd") or props.get("eo:gsd", 0)
        item_geom = shape(item["geometry"]) if item.get("geometry") else None

        scenes.append({
            "id": item["id"],
            "url": asset["href"],
            "cloud_cover": float(cc) if cc is not None else None,
            "datetime": props.get("datetime"),
            "footprint": item_geom,
            "resolution": float(gsd) if gsd else None,
            "bbox": item.get("bbox"),
        })

    # Sort by cloud cover ascending (same as cog_service)
    if cc_field:
        scenes.sort(key=lambda s: (s["cloud_cover"] if s["cloud_cover"] is not None else 999, s["datetime"] or ""))

    return scenes


# Try importing the actual backend function first, fall back to direct HTTP
if COG_SERVICE_AVAILABLE:
    print("Using backend _query_stac_scenes()...")
    try:
        scenes = _query_stac_scenes(
            SOURCE_ID, wajir_geometry, BUFFER_METERS,
            DATE_FROM, DATE_TO, MAX_CLOUD_COVER,
        )
        scenes_dict = [{
            "id": s.id,
            "url": s.url,
            "cloud_cover": s.cloud_cover,
            "datetime": s.datetime,
            "footprint": s.footprint,
            "resolution": s.resolution,
        } for s in scenes]
        print(f"Found {len(scenes_dict)} scenes via backend")
    except Exception as e:
        print(f"Backend query failed: {e}")
        print("Falling back to direct HTTP query...")
        scenes_dict = query_stac_scenes_direct(
            SOURCE_ID, wajir_geometry, BUFFER_METERS,
            DATE_FROM, DATE_TO, MAX_CLOUD_COVER,
        )
else:
    print("Using direct HTTP STAC query...")
    scenes_dict = query_stac_scenes_direct(
        SOURCE_ID, wajir_geometry, BUFFER_METERS,
        DATE_FROM, DATE_TO, MAX_CLOUD_COVER,
    )

In [ ]:
if scenes_dict:
    print(f"{'Scene ID':<60} {'Date':<25} {'Cloud%':<8} {'Res':<6} {'Has URL':<10}")
    print("-" * 110)
    for s in scenes_dict:
        dt = s.get("datetime", "")[:19] if s.get("datetime") else "N/A"
        cc = f"{s['cloud_cover']:.0f}%" if s.get("cloud_cover") is not None else "N/A"
        res = f"{s['resolution']}m" if s.get("resolution") else "N/A"
        has_url = "✓" if s.get("url") else "✗"
        print(f"{s['id']:<60} {dt:<25} {cc:<8} {res:<6} {has_url:<10}")
else:
    print("No scenes found for Wajir with the current filter parameters.")
    print("Try expanding the date range or increasing max_cloud_cover.")

---
## 4. Coverage Validation

This is what `_check_scene_coverage()` does in `cog_service.py`: it unions all scene footprints and checks whether they **contain** the target polygon (or at least cover ≥80% of it).

If coverage is insufficient, the API returns a `CoverageError` telling the user to expand dates or raise cloud cover.

In [ ]:
def check_coverage(scenes: list[dict], target_poly) -> tuple[bool, float | None]:
    """Same logic as _check_scene_coverage in cog_service.py."""
    if not scenes:
        return False, None
    footprints = [s.get("footprint") for s in scenes if s.get("footprint") is not None]
    if not footprints or target_poly.area == 0:
        return False, None
    combined = unary_union(footprints)
    overlap = combined.intersection(target_poly).area
    coverage_pct = overlap / target_poly.area * 100
    return combined.contains(target_poly), coverage_pct


contained, coverage_pct = check_coverage(scenes_dict, wajir_poly)

print(f"Coverage analysis for Wajir:")
print(f"  Polygon area:         {wajir_poly.area:.6f} sq.deg")
print(f"  Union of footprints:  {unary_union([s['footprint'] for s in scenes_dict if s.get('footprint')]).area:.6f} sq.deg" if scenes_dict else "  No footprints")
print(f"  Coverage percentage:  {coverage_pct:.1f}%" if coverage_pct is not None else "  N/A")
print(f"  Fully contained:      {'✓ YES' if contained else '✗ NO'}")

if coverage_pct is not None and coverage_pct < 80:
    print(f"\n  ⚠ WARNING: Coverage below 80% threshold.")
    print(f"     The backend would raise a CoverageError.")
    print(f"     Recommendation: expand date range or raise cloud cover limit.")

### Visualize scene footprints vs. Wajir

In [ ]:
if HAS_MPL and scenes_dict:
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))

    # Plot Wajir boundary
    if wajir_poly.geom_type == "MultiPolygon":
        for poly in wajir_poly.geoms:
            xs, ys = poly.exterior.xy
            ax.fill(xs, ys, alpha=0.2, fc="blue", ec="darkblue", lw=2, label="Wajir boundary")
    else:
        xs, ys = wajir_poly.exterior.xy
        ax.fill(xs, ys, alpha=0.2, fc="blue", ec="darkblue", lw=2, label="Wajir boundary")

    # Plot scene footprints
    colors = ["red", "green", "orange", "purple", "brown", "pink", "olive", "cyan"]
    for i, s in enumerate(scenes_dict):
        fp = s.get("footprint")
        if fp is None:
            continue
        if fp.geom_type == "Polygon":
            xs, ys = fp.exterior.xy
            ax.plot(xs, ys, color=colors[i % len(colors)], lw=1, linestyle="--", alpha=0.7)
        elif fp.geom_type == "MultiPolygon":
            for part in fp.geoms:
                xs, ys = part.exterior.xy
                ax.plot(xs, ys, color=colors[i % len(colors)], lw=1, linestyle="--", alpha=0.7)
        # Label the best scene
        if i == 0:
            cx, cy = fp.centroid.x, fp.centroid.y
            ax.annotate(f"Best: {s['id'][:30]}...", (cx, cy), fontsize=7, color=colors[i % len(colors)])

    ax.set_title(f"Wajir + {len(scenes_dict)} Sentinel-2 scene footprints")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)
    plt.show()
else:
    print("Matplotlib not available or no scenes to plot.")

---
## 5. Phase 1 — Preview

This is what `get_preview_scene()` returns instantly (~1s). It selects the best scene (lowest cloud cover, most recent) and returns its COG URL as `preview_url`.

The frontend renders this via TiTiler at:
```
/cog/tiles/WebMercatorQuad/{z}/{x}/{y}@1x.png?url={preview_url}&geojson={polygon_geojson}
```
The `&geojson=` parameter tells TiTiler to **clip the raster rendering** to the polygon shape **on-the-fly** — no download needed.

In [ ]:
if scenes_dict:
    best = scenes_dict[0]
    print(f"Best scene:")
    print(f"  ID:          {best['id']}")
    print(f"  Date:        {best['datetime']}")
    print(f"  Cloud cover: {best['cloud_cover']:.1f}%" if best.get('cloud_cover') is not None else "  Cloud cover: N/A")
    print(f"  Resolution:  {best['resolution']}m" if best.get('resolution') else "  Resolution:  N/A")
    print(f"\nPreview URL (raw COG asset):")
    print(f"  {best['url']}")

    # Build the TiTiler URL like ImageryModal does
    import urllib.parse
    preview_url_encoded = urllib.parse.quote(best['url'], safe='')
    geojson_str = json.dumps(wajir_geometry)
    geojson_encoded = urllib.parse.quote(geojson_str, safe='')
    titiler_url = f"/cog/tiles/WebMercatorQuad/{{z}}/{{x}}/{{y}}@1x.png?url={preview_url_encoded}&geojson={geojson_encoded}"
    print(f"\nTiTiler rendering URL (as used by ImageryModal):")
    print(f"  {titiler_url[:200]}...")

    # Area calculation (same as compute_geojson_area)
    area_km2 = wajir_poly.area
    centroid = wajir_poly.centroid
    cos_lat = abs(math.cos(math.radians(centroid.y))) if centroid.y is not None else 1.0
    area_km2_real = area_km2 * (111_320 ** 2) * cos_lat / 1_000_000
    print(f"\nScene statistics:")
    print(f"  Area (approx): {area_km2_real:,.2f} km²")
    print(f"  Scene count:   {len(scenes_dict)}")
    dates = [s['datetime'] for s in scenes_dict if s.get('datetime')]
    if dates:
        print(f"  Date range:    {min(dates)[:10]} → {max(dates)[:10]}")
    cloud_covers = [s['cloud_cover'] for s in scenes_dict if s.get('cloud_cover') is not None]
    if cloud_covers:
        print(f"  Avg cloud:     {sum(cloud_covers)/len(cloud_covers):.1f}%")

    print(f"\n✓ Phase 1 preview is available. The COG is already a valid Cloud-Optimized GeoTIFF.")
    print(f"  TiTiler can render it clipped to the polygon without any preprocessing.")
else:
    print("No scenes available for preview.")

---
## 6. Phase 2 — Clipped COG Generation (Background)

After Phase 1 returns the preview, the backend fires Phase 2 in a background thread. This:
1. Calls `_query_stac_scenes()` to get all matching scenes (same as Phase 1)
2. Downloads or streams each scene via `/vsicurl/`
3. If multiple scenes, builds a VRT mosaic with `gdalbuildvrt`
4. Clips to polygon+buffer with `gdalwarp -cutline -crop_to_cutline -dstalpha`
5. Saves as COG and uploads to GCS
6. Inserts into `cog_catalog` table for spatial indexing

**Important:** For STAC sources, the raw asset is **already a COG**. Phase 2 is redundant for display — it exists for:
- **Download**: user gets a GeoTIFF clipped exactly to their polygon
- **Caching**: subsequent requests are instant from GCS
- **Mosaicing**: when multiple scenes are needed, they're merged into one COG

In [ ]:
# Show the actual gdalwarp command that Phase 2 uses
print("=== Phase 2: gdalwarp clip command ===")
print("")
print("This is what _generate_clipped_cog() runs (cog_service.py:662-710):")
print("")
print("# 1. Write cutline to GeoJSON")
print("cutline_path = 'cutline.geojson'")
print("")
print("# 2. Clip the VRT mosaic to polygon+buffer, output as COG")
print("gdalwarp input_path output_path \\")
print("    -of COG \\")
print("    -t_srs EPSG:4326 \\")
print("    -cutline cutline_path \\")
print("    -crop_to_cutline \\")
print("    -dstalpha \\")
print("    -co COMPRESS=DEFLATE \\")
print("    -co BLOCKSIZE=512 \\")
print("    -r bilinear \\")
print("    -wm 1024 \\")
print("    -overwrite")
print("")
print("# 3. Fallback: warp then translate")
print("gdalwarp ... -t_srs EPSG:4326 ...  # if direct COG output fails")
print("gdal_translate ... -of COG -co RESAMPLING=LANCZOS")

In [ ]:
if len(scenes_dict) > 1:
    print("=== VRT Mosaic (gdalbuildvrt) ===")
    print("")
    print("When multiple scenes are needed to cover the polygon:")
    print("")
    print("gdalbuildvrt mosaic.vrt \\")
    print("    -input_file_list scenes.txt \\")
    print("    -t_srs EPSG:4326 \\")
    print("    -overwrite")
    print("")
    print("This merges all scenes into a single virtual raster before clipping.")
elif len(scenes_dict) == 1:
    print("Only 1 scene covers the area — no mosaic needed.")
    print("The raw scene is used directly as input to gdalwarp.")
else:
    print("No scenes to mosaic.")

---
## 7. Preprocessing Audit

What satellite preprocessing steps are currently implemented vs. what's missing in the imagery pipeline.

In [ ]:
audit = [
    ("Date range filtering", "✅", "STAC `datetime` parameter filters scenes by acquisition date"),
    ("Cloud cover (metadata)", "✅", "STAC `eo:cloud_cover` query filter; scenes sorted by cloud cover"),
    ("Scene coverage check", "✅", "≥80% footprint union coverage required; else CoverageError"),
    ("Scene sorting/ranking", "✅", "Sort by cloud cover asc → datetime desc for best-first"),
    ("Clipping to polygon+buffer", "✅", "gdalwarp -cutline -crop_to_cutline with alpha band"),
    ("Multi-scene mosaicing", "✅", "gdalbuildvrt merges overlapping scenes into single VRT"),
    ("COG optimization", "✅", "DEFLATE compression, 512px blocks, bilinear resampling"),
    ("GCS caching", "✅", "COG + metadata cached by source_id + geometry_hash; cog_catalog spatial"),
    ("", "", ""),
    ("Pixel-level cloud masking", "✅", "SCL band (Sentinel-2) + QA_PIXEL (Landsat) + s2cloudless (opt-in ML)"),
    ("Atmospheric correction", "❌", "Not applied (acceptable for L2A/SR sources; gap for L1C)"),
    ("Pan-sharpening", "✅", "Brovey / PCA / IHS (user-selectable, requires pan band)"),
    ("Radiometric normalization", "❌", "No inter-scene radiometric balancing in mosaics"),
    ("Topographic correction", "❌", "No terrain illumination correction"),
    ("BRDF normalization", "❌", "No viewing-angle correction for mosaics"),
    ("Band index computation", "❌", "No NDVI/NDWI/etc. derived products"),
]

print(f"{'Step':<35} {'Status':<8} {'Detail'}")
print("=" * 100)
for step, status, detail in audit:
    if not step:
        print("-" * 100)
    else:
        print(f"{step:<35} {status:<8} {detail}")

---
## 8. All Satellite Sources

Summary of all providers currently plugged into the backend and frontend.

In [ ]:
# Source labels and coverage (hardcoded here since frontend TS files aren't Python-importable)
FRONTEND_SOURCES = {
    "naip":              {"label": "NAIP (60cm, US)", "coverage": "Continental US"},
    "sentinel-2-l2a":    {"label": "Sentinel-2 L2A (10m, Global)", "coverage": "Global"},
    "landsat-c2-l2":     {"label": "Landsat 8/9 (30m, Global)", "coverage": "Global"},
    "sentinel-1-rtc":    {"label": "Sentinel-1 SAR (10m, Global)", "coverage": "Global"},
    "cop-dem-glo-30":    {"label": "Copernicus DEM (30m, Global)", "coverage": "Global"},
    "cbers-4-mux":       {"label": "CBERS-4 (64m, Latin America)", "coverage": "Latin America"},
    "amazonia-1-wfi":    {"label": "Amazonia-1 WFI (64m, Brazil)", "coverage": "Brazil"},
}

XYZ_SOURCES = {
    "esri-world-imagery": "Esri World Imagery (~1m, Global)",
    "osm":                "OpenStreetMap (Reference)",
}

COMMERCIAL_SOURCES = {
    "planet":  {"provider": "planet"},
    "maxar":   {"provider": "maxar"},
    "airbus":  {"provider": "airbus"},
    "blacksky":{"provider": "blacksky"},
}

print(f"{'ID':<22} {'Type':<12} {'Collection/Provider':<28} {'Label':<32} {'STAC URL'}")
print("=" * 120)

for sid, cfg in SOURCE_CONFIG.items():
    col = cfg.get("collection", "-")
    url = cfg.get("stac_url", "-")
    label = FRONTEND_SOURCES.get(sid, {}).get("label", "-")
    print(f"{sid:<22} {cfg['type']:<12} {col:<28} {label:<32} {url}")

print()
print("Commercial sources (require API key / credits):")
for sid, cfg in COMMERCIAL_SOURCES.items():
    print(f"  {sid:<20} type=commercial  provider={cfg['provider']}")

print()
print("Frontend-only tile sources (no COG fetch — XYZ tiles on map):")
for sid, label in XYZ_SOURCES.items():
    print(f"  {sid:<22} {label}")

---
## Summary: End-to-End Flow

```
User clicks admin boundary on map
        │
        ▼
ImageryModal opens
  • Bold blue outline of polygon (fill at 15%, line-width 2)
  • Amber dashed buffer zone (if buffer > 0)
  • Date range + cloud cover + source selectors
        │
        ▼ "Fetch Satellite Data" clicked
POST /api/imagery/fetch
        │
        ├── Phase 1 (sync, ~1s)
        │     • _query_stac_scenes() — STAC query with filters
        │     • _check_scene_coverage() — ≥80% coverage required
        │     • Returns preview_url (raw STAC COG asset URL)
        │     • Frontend renders via TiTiler + &geojson= clip
        │
        └── Phase 2 (background thread)
              • Downloads/streams scenes via /vsicurl/
              • gdalbuildvrt (if multiple scenes)
              • gdalwarp -cutline -crop_to_cutline -dstalpha
              • Uploads clipped COG to GCS
              • Inserts into cog_catalog
        
Frontend polls GET /api/imagery/fetch/status/{job_hash}
        │
        ▼ When "ready"
Swaps TiTiler preview → pre-clipped COG URL
User can download GeoTIFF or export PNG/JPEG
```

**Key takeaway:** For STAC sources (NAIP, Sentinel-2, Landsat, etc.), the asset URL from STAC **is already a COG**. The TiTiler + `&geojson=` approach renders it clipped on-the-fly for display. Phase 2 re-downloads and re-clips only to produce a standalone cached file for download and caching.

---
## Source Code Reference

Key files referenced throughout this notebook:

In [ ]:
files = {
    "cog_service.py (full pipeline)": "../backend/engine/service_layer/cog_service.py",
    "api.py (imagery fetch endpoints)": "../backend/engine/entrypoints/api.py",
    "ImageryModal.tsx (frontend UI)": "../frontend/src/components/Imagery/ImageryModal.tsx",
    "ImageryFetchRequest/Response types": "../frontend/src/types/polygon.ts",
    "Source config / IMAGERY_SOURCES": "../frontend/src/types/polygon.ts",
}

print("Key files:")
for name, path in files.items():
    exists = os.path.exists(os.path.join(PROJECT_ROOT, "..", path) if path.startswith("..") else os.path.join(PROJECT_ROOT, path))
    full_path = os.path.abspath(os.path.join(os.getcwd(), path))
    print(f"  {name:<40} {full_path}")